# 04. Abituriyentlar ma'lumotlarini tayyorlash (Data Preparation)

`03`-notebookdagi xulosalar asosida `real_abts.csv` ni tozalaymiz:

1. Fan nomlarini bir xillashtirish (`Kasbiy (ijodiy) imtihon`, `Oʻzbek tili va adabiyoti`)
2. `fanlar_juftligi` ustunini yaratish
3. 189 dan yuqori ballar uchun belgi qo'shish
4. `ismi` (shaxsiy ma'lumot) ustunini olib tashlash
5. Tekshirish va `data/processed/abts_prepared.csv` ga saqlash

**Data leakage:** `abt_unique.csv` dagi OTM, yo'nalish, til va shakl ustunlari mandat chiqqandan keyin yig'ilgan — ular abituriyent qayerga kirganini ko'rsatadi. Hozirgi qabul tartibida abituriyent avval ballini bilib, keyin yo'nalish tanlaydi, shuning uchun bashorat paytida bu ma'lumotlar bo'lmaydi. Ularni jadvalga qo'shmaymiz: faqat abituriyent oldindan biladigan ma'lumotlar — ball va imtihon fanlari qoladi.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/processed/real_abts.csv')
print(df.shape)
df.head()

(344701, 5)


,ismi,ID,toplagan_bali,1-fan,2-fan
0,URAKBAYEV N. M.,6042695,189.0,Matematika,Fizika
1,NAZEROV I. A.,6045467,189.0,Matematika,Fizika
2,TENGELOV X. I.,6872807,189.0,Matematika,Fizika
3,JANABAYEV S. O.,6259423,186.7,Matematika,Fizika
4,ERDOSHEVA X. T.,6980903,185.7,Matematika,Fizika


## 1. Fan nomlarini bir xillashtirish

- `Kasbiy (ijodiy) imtihon` va `Kasbiy (ijodiy imtihon)` — bitta fan (`03`-notebookda topilgan).
- `Oʻzbek tili va adabiyoti` va `Ona tili va adabiyoti` — bitta fan, `Ona tili va adabiyoti` deb yoziladi.

In [3]:
fan_tuzatish = {
    'Kasbiy (ijodiy) imtihon': 'Kasbiy (ijodiy imtihon)',
    'Oʻzbek tili va adabiyoti': 'Ona tili va adabiyoti',
}

for col in ['1-fan', '2-fan']:
    df[col] = df[col].replace(fan_tuzatish)
    print(f"{col}: {df[col].nunique()} ta noyob qiymat")

1-fan: 19 ta noyob qiymat
2-fan: 9 ta noyob qiymat


## 2. `fanlar_juftligi` ustuni

Juftlik fanlar tartibini saqlaydi (`Matematika + Fizika` ≠ `Fizika + Matematika`). Lekin bu raqobat birligi emas: abituriyentning bali hamma tanlovlarida bir xil va bir nechta juftlikka hujjat topshira oladi. Raqobat birligi — `06`-notebookda aniqlanadigan **bozor**.

In [4]:
df['fanlar_juftligi'] = df['1-fan'] + ' + ' + df['2-fan']
print('Noyob juftliklar:', df['fanlar_juftligi'].nunique())
df['fanlar_juftligi'].value_counts().head(10)

Noyob juftliklar: 37


fanlar_juftligi
Matematika + Chet tili                               57532
Biologiya + Kimyo                                    47518
Matematika + Fizika                                  44177
Biologiya + Ona tili va adabiyoti                    30179
Ingliz tili + Ona tili va adabiyoti                  25345
Ona tili va adabiyoti + Chet tili                    24605
Huquqshunoslik fanlari + Chet tili                   16037
Tarix + Chet tili                                    14923
Ona tili va adabiyoti + Matematika                   13497
Kasbiy (ijodiy imtihon) + Kasbiy (ijodiy imtihon)    10649
Name: count, dtype: int64

## 3. 189 dan yuqori ballar

Test bo'yicha maksimal ball 189. Undan yuqori ballar (111 ta) ehtimol qo'shimcha ball (sertifikat, imtiyoz) hisobiga. Qiymatni o'zgartirmaymiz, faqat belgi qo'shamiz — keyingi bosqichda ularni cheklash yoki alohida ko'rish mumkin.

In [5]:
MAKS_TEST_BALI = 189

df['qoshimcha_ball'] = df['toplagan_bali'] > MAKS_TEST_BALI
df['qoshimcha_ball'].value_counts()

qoshimcha_ball
False    344590
True        111
Name: count, dtype: int64

## 4. Keraksiz ustunlarni olib tashlash va tartiblash

`ismi` — shaxsiy ma'lumot, tahlil va model uchun kerak emas. `ID` qoladi: u orqali boshqa jadvallar bilan bog'lash mumkin.

In [6]:
df = df.drop(columns='ismi')

df = df[['ID', '1-fan', '2-fan', 'fanlar_juftligi', 'toplagan_bali', 'qoshimcha_ball']]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344701 entries, 0 to 344700
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   ID               344701 non-null  int64  
 1   1-fan            344701 non-null  str    
 2   2-fan            344701 non-null  str    
 3   fanlar_juftligi  344701 non-null  str    
 4   toplagan_bali    344701 non-null  float64
 5   qoshimcha_ball   344701 non-null  bool   
dtypes: bool(1), float64(1), int64(1), str(3)
memory usage: 13.5 MB


## 5. Yakuniy tekshiruv va saqlash

In [7]:
assert df['ID'].is_unique
assert df.isna().sum().sum() == 0, df.isna().sum()[lambda s: s > 0]
assert df['1-fan'].nunique() == 19 and df['2-fan'].nunique() == 9
assert not set(fan_tuzatish) & (set(df['1-fan']) | set(df['2-fan']))
print('Barcha tekshiruvlardan o\'tdi:', df.shape)

Barcha tekshiruvlardan o'tdi: (344701, 6)


In [8]:
df.to_csv('../data/processed/abts_prepared.csv', index=False, encoding='utf-8')
print('Saqlandi: data/processed/abts_prepared.csv')

Saqlandi: data/processed/abts_prepared.csv


## Natija

`abts_prepared.csv` — 344 701 qator, 6 ustun, bo'sh qiymatlarsiz. Har bir qator: abituriyent `ID`, imtihon fanlari, fanlar juftligi, to'plangan ball va qo'shimcha ball belgisi.

**Eslatma:** bu jadvalda `01`-notebookda fanlari topilmagan 12 860 abituriyent (asosan sport va san'at yo'nalishlari) yo'q. To'liq nomzodlar to'plami (357 561) `07`-notebookda `abiturents.csv` va `06` dagi raqobat bozorlari orqali quriladi.

### Keyingi qadamlar
- Model uchun kategorial ustunlarni (`1-fan`, `2-fan`, `fanlar_juftligi`) kodlash.